# Router Agent
The router agent decides whether to use the SQL agent or the Semantic Search agent based on the user's query.

In [1]:
import sys
sys.path.insert(0, "..")

from movie_agent.agent.router import ask, ask_stream

d:\Personal\arrow_task\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model from local: d:\Personal\arrow_task\notebooks\..\models\all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3269.86it/s]

Model loaded.


## Non-streaming: `ask()`

In [3]:
# Factual query -> should route to SQL agent
result = ask("How many movies are in the database?")
print(result["answer"])
print(f"\nSession ID: {result['session_id']}")

There are a total of 4,803 movies in the database.

Session ID: 793e4b39-7340-4a8a-9258-2ae00d24bac6


In [4]:
# Descriptive query -> should route to semantic search agent
result = ask("I want a movie about time travel and love")
print(result["answer"])

Here are some great movies that blend time travel with love:

1. **The Lovers** - An epic romance time travel adventure that tells a sweeping tale of impossible love set against a historical backdrop.

2. **Boyhood** - While not a traditional time travel movie, it captures the essence of time passing through the life of a boy growing up, exploring relationships, including love, over twelve years.

3. **About Time** - This charming film follows Tim, who learns he can travel through time. He uses this ability to improve his love life, leading to heartfelt moments and lessons about love and family.

4. **Kate & Leopold** - A romantic comedy where a modern woman named Kate finds herself in a love story with a 19th-century nobleman named Leopold, who travels through time.

5. **The Quiet American** - More of a political thriller, it features a love triangle set against the backdrop of 1950s Vietnam, exploring the complexities of love in a tumultuous time.

If any of these catch your interes

In [ ]:
# Mixed query -> might use both tools
result = ask("Recommend a highly-rated sci-fi movie about artificial intelligence")
print(result["answer"])

## Streaming: `ask_stream()`
See each step the router takes: which sub-agent it calls, the result, and the final answer.

In [7]:
# Streaming - factual query
print("=" * 60)
print("Query: What are the top 5 highest rated movies?")
print("=" * 60)

for event in ask_stream("What are the top 5 highest rated movies?"):
    if event["type"] == "tool_call":
        print(f"\n\U0001f6e0\ufe0f  ROUTED TO: {event['tool']}")
        print(f"   Input: {event['input']}")
    elif event["type"] == "tool_result":
        print(f"\n\U0001f4cb RESULT from {event['tool']}:")
        print(f"   {event['output'][:500]}")
    elif event["type"] == "token":
        print(event["content"], end="", flush=True)

print()

Query: What are the top 5 highest rated movies?

🛠️  ROUTED TO: sql_agent_tool
   Input: {'question': 'Top 5 highest rated movies'}

📋 RESULT from sql_agent_tool:
   Here are the top 5 highest-rated movies based on their average votes:

1. **Little Big Top** - Rating: 10.0
2. **Dancer, Texas Pop. 81** - Rating: 10.0
3. **Stiff Upper Lips** - Rating: 10.0
4. **Me You and Five Bucks** - Rating: 10.0
5. **Sardaarji** - Rating: 9.5

It's interesting to see that several movies share the perfect rating of 10.0!
Here are the top 5 highest-rated movies based on their average votes:

1. **Little Big Top** - Rating: 10.0
2. **Dancer, Texas Pop. 81** - Rating: 10.0
3. **Stiff Upper Lips** - Rating: 10.0
4. **Me You and Five Bucks** - Rating: 10.0
5. **Sardaarji** - Rating: 9.5

It's notable that several movies share the perfect rating of 10.0!


In [8]:
# Streaming - descriptive query
print("=" * 60)
print("Query: Find me a dark psychological thriller with a twist")
print("=" * 60)

session_id = None
for event in ask_stream("Find me a dark psychological thriller with a twist"):
    session_id = event["session_id"]
    if event["type"] == "tool_call":
        print(f"\n\U0001f6e0\ufe0f  ROUTED TO: {event['tool']}")
        print(f"   Input: {event['input']}")
    elif event["type"] == "tool_result":
        print(f"\n\U0001f4cb RESULT from {event['tool']}:")
        print(f"   {event['output'][:500]}")
    elif event["type"] == "token":
        print(event["content"], end="", flush=True)

print(f"\n\nSession ID: {session_id}")

Query: Find me a dark psychological thriller with a twist

🛠️  ROUTED TO: semantic_search_tool
   Input: {'query': 'dark psychological thriller with a twist'}

📋 RESULT from semantic_search_tool:
   Here are some dark psychological thrillers with intriguing twists that you might enjoy:

1. **The Sixth Sense** (Rating: 7.7)
   - This classic psychological thriller follows an eight-year-old boy named Cole Sear, who believes he can see dead people. A child psychologist, Malcolm Crowe, tries to help him, leading to a shocking twist that redefines the entire story. It's a masterclass in suspense and emotional depth.

2. **The Horror Network Vol. 1** (Rating: 5.0)
   - This anthology features a c
Here are some dark psychological thrillers with intriguing twists that you might enjoy:

1. **The Sixth Sense** (Rating: 7.7)
   - This classic psychological thriller follows an eight-year-old boy named Cole Sear, who believes he can see dead people. A child psychologist, Malcolm Crowe, tries to hel

In [ ]:
# Follow-up with memory (same session)
print("Follow-up: Tell me more about the first one\n")

for event in ask_stream("Tell me more about the first one", session_id=session_id):
    if event["type"] == "tool_call":
        print(f"\n\U0001f6e0\ufe0f  ROUTED TO: {event['tool']}")
        print(f"   Input: {event['input']}")
    elif event["type"] == "tool_result":
        print(f"\n\U0001f4cb RESULT from {event['tool']}:")
        print(f"   {event['output'][:500]}")
    elif event["type"] == "token":
        print(event["content"], end="", flush=True)

print()